# PDHG-HJ for Total-Variation deblurring

Compares the Chambolle–Pock primal–dual hybrid-gradient algorithm using the analytical TV proximal against PDHG with HJ-Prox.  Reproduces a panel of Figure 1.

## Setup


In [ ]:
# ============================================================================
# CHUNK 1: SETUP - Algorithms, Helper Functions, and Definitions
# ============================================================================

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.signal import convolve2d
from hj_prox import hj_prox
import time
from typing import Tuple

EPS = 1e-5

# Set deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Plotting configuration
plt.rcParams.update({'font.size': 16})


# ============================================================================
# Helper Functions
# ============================================================================

def create_gaussian_kernel(size, sigma):
    """Create a 2D Gaussian kernel for blurring."""
    kernel = np.zeros((size, size), dtype=float)
    center = size // 2
    for i in range(size):
        for j in range(size):
            x, y = i - center, j - center
            kernel[i, j] = np.exp(-(x**2 + y**2) / (2 * sigma**2))
    return kernel / kernel.sum()


def blur_image(img, kernel_size=9, sigma=2.0):
    """Apply Gaussian blur to an image."""
    kernel = create_gaussian_kernel(kernel_size, sigma)
    blurred = convolve2d(img, kernel, mode='same', boundary='symm')
    return blurred, kernel


def compute_tv(x: torch.Tensor) -> float:
    """Compute Total Variation of an image using forward differences."""
    dx = torch.zeros_like(x)
    dy = torch.zeros_like(x)
    
    # Forward differences
    dx[:-1, :] = x[1:, :] - x[:-1, :]
    dy[:, :-1] = x[:, 1:] - x[:, :-1]
    
    # TV norm
    tv = torch.sqrt(dx**2 + dy**2 + 1e-8)
    return tv.sum().item()


def compute_tv_pytorch(x_batch: torch.Tensor, H: int, W: int) -> torch.Tensor:
    """Compute Total Variation for a batch of images."""
    batch_size = x_batch.shape[0]
    x_imgs = x_batch.view(batch_size, H, W)
    dx = torch.zeros_like(x_imgs)
    dy = torch.zeros_like(x_imgs)
    dx[:, :-1, :] = x_imgs[:, 1:, :] - x_imgs[:, :-1, :]
    dy[:, :, :-1] = x_imgs[:, :, 1:] - x_imgs[:, :, :-1]
    # Enforce zero gradient at bottom/right
    dx[:, -1, :] = 0
    dy[:, :, -1] = 0
    tv = torch.sqrt(dx**2 + dy**2 + 1e-8)
    return tv.sum(dim=(1,2))


# ============================================================================
# Algorithm 1: PDHG (Chambolle-Pock) with Analytical Operators
# ============================================================================

def chambolle_pock_tv_deblur(
    blurred_img: np.ndarray,
    kernel: np.ndarray,
    lambda_1: float = 0.02,
    iterations: int = 500,
    tau: float = None,
    sigma: float = None,
    theta: float = 1.0,
    verbose: bool = True,
    device: str = 'cpu'
) -> Tuple[np.ndarray, list]:
    """
    Deblur image using TV lambda_1 with Chambolle-Pock algorithm.
    
    Solves: minimize 0.5 * ||K*x - y||^2 + λ * TV(x)
    
    Args:
        blurred_img: Blurred input image
        kernel: Blur kernel
        lambda_1: TV lambda_1 parameter (λ)
        iterations: Number of iterations
        tau: Primal step size
        sigma: Dual step size
        theta: Extrapolation parameter (typically 1.0)
        verbose: Print progress
        device: 'cpu' or 'cuda'
    
    Returns:
        Deblurred image and list of objective values
    """
    H, W = blurred_img.shape
    y = torch.from_numpy(blurred_img).float().to(device)
    
    # Initialize primal variable
    x = y.clone()
    x_bar = y.clone()
    
    # Initialize dual variables
    z = torch.zeros_like(y)  # Dual for data fidelity
    p1 = torch.zeros_like(y)  # Dual for TV (gradient x)
    p2 = torch.zeros_like(y)  # Dual for TV (gradient y)
    
    # Prepare kernel for FFT convolution
    kernel_torch = torch.from_numpy(kernel).float().to(device)
    kernel_padded = torch.zeros(H, W, device=device)
    kh, kw = kernel.shape
    kernel_padded[:kh, :kw] = kernel_torch
    
    # Center the kernel for FFT
    kernel_padded = torch.roll(kernel_padded, shifts=(-kh//2, -kw//2), dims=(0, 1))
    kernel_fft = torch.fft.fft2(kernel_padded)
    kernel_fft_conj = torch.conj(kernel_fft)
    
    # Set step sizes if not provided
    if tau is None or sigma is None:
        L = 12.0  # Conservative estimate of operator norm
        tau = 0.9 / L
        sigma = 0.9 / L
        if verbose:
            print(f"Using step sizes: tau={tau:.4f}, sigma={sigma:.4f}")
    
    losses = []
    
    for it in range(iterations):
        x_old = x.clone()
        
        # ---- Dual update ----
        # Update z (data fidelity dual)
        x_bar_fft = torch.fft.fft2(x_bar)
        Kx_bar = torch.real(torch.fft.ifft2(x_bar_fft * kernel_fft))
        z = (z + sigma * (Kx_bar - y)) / (1.0 + sigma)
        
        # Update p (TV dual) with forward differences
        grad_x = torch.zeros_like(x_bar)
        grad_y = torch.zeros_like(x_bar)
        grad_x[:-1, :] = x_bar[1:, :] - x_bar[:-1, :]
        grad_y[:, :-1] = x_bar[:, 1:] - x_bar[:, :-1]
        
        p1 = p1 + sigma * grad_x
        p2 = p2 + sigma * grad_y
        
        # Project onto L∞ ball
        norm = torch.sqrt(p1**2 + p2**2 + 1e-8)
        factor = torch.clamp(lambda_1 / norm, max=1.0)
        p1 = p1 * factor
        p2 = p2 * factor
        
        # ---- Primal update ----
        # K^T z
        z_fft = torch.fft.fft2(z)
        KTz = torch.real(torch.fft.ifft2(z_fft * kernel_fft_conj))
        
        # Divergence of p (negative adjoint of gradient)
        div_p = torch.zeros_like(x)
        # Backward differences
        div_p[1:, :] -= p1[:-1, :]
        div_p[:-1, :] += p1[:-1, :]
        div_p[:, 1:] -= p2[:, :-1]
        div_p[:, :-1] += p2[:, :-1]
        
        # Update x
        x = x - tau * (KTz - div_p)
        
        # Extrapolation
        x_bar = x + theta * (x - x_old)
        
        # Monitor convergence
        if it % 1 == 0:
            x_fft = torch.fft.fft2(x)
            Kx = torch.real(torch.fft.ifft2(x_fft * kernel_fft))
            
            data_term = 0.5 * torch.sum((Kx - y)**2).item()
            tv_term = compute_tv(x)
            loss = data_term + lambda_1 * tv_term
            losses.append(loss)
            
            if verbose and it % 50 == 0:
                primal_res = torch.norm(x - x_old).item()
                print(f"Iter {it}: Loss={loss:.6f}, Data={data_term:.6f}, "
                      f"TV={tv_term:.6f}, ||Δx||={primal_res:.6f}")
    
    return x.cpu().numpy(), losses


# ============================================================================
# Algorithm 2: PDHG with HJ-Prox
# ============================================================================

def pdhg_hjprox_deblur(
    blurred_img: np.ndarray,
    kernel: np.ndarray,
    lambda_1: float = 0.02,
    iterations: int = 500,
    tau: float = None,
    sigma: float = None,
    theta: float = 1.0,
    num_samples: int = 100,
    delta: float = 1e-2,
    verbose: bool = True,
    device: str = 'cpu',
    init_img: np.ndarray = None
) -> Tuple[np.ndarray, list]:
    """
    Deblur image using PDHG with HJ-Prox for TV lambda_1.
    
    Solves: minimize 0.5 * ||K*x - y||^2 + λ * TV(x)
    
    Uses HJ-Prox to compute the proximal operator of TV.
    
    Args:
        blurred_img: Blurred input image
        kernel: Blur kernel
        lambda_1: TV lambda_1 parameter (λ)
        iterations: Number of iterations
        tau: Primal step size
        sigma: Dual step size
        theta: Extrapolation parameter
        num_samples: Number of samples for HJ-Prox
        delta: Smoothing parameter for HJ-Prox
        verbose: Print progress
        device: 'cpu' or 'cuda'
        init_img: Initial image (if None, uses blurred_img)
    
    Returns:
        Deblurred image and list of objective values
    """
    H, W = blurred_img.shape
    y = torch.from_numpy(blurred_img).float().to(device)
    
    # Initialize x and x_bar with provided image or default to blurred image
    if init_img is not None:
        if init_img.shape != blurred_img.shape:
            raise ValueError(f"init_img shape {init_img.shape} must match blurred_img shape {blurred_img.shape}")
        x = torch.from_numpy(init_img).float().to(device)
        x_bar = x.clone()
    else:
        x = y.clone()
        x_bar = y.clone()
    
    z = torch.zeros_like(y)
    
    # Prepare kernel for convolution via FFT
    kernel_torch = torch.from_numpy(kernel).float().to(device)
    kernel_padded = torch.zeros(H, W, device=device)
    kh, kw = kernel.shape
    kernel_padded[:kh, :kw] = kernel_torch
    # Center before FFT to ensure correct alignment
    kernel_padded = torch.roll(kernel_padded, shifts=(-kh//2, -kw//2), dims=(0, 1))
    kernel_fft = torch.fft.fft2(kernel_padded)
    kernel_fft_conj = torch.conj(kernel_fft)
    
    # Set step sizes if not provided
    if tau is None or sigma is None:
        L = 3.0
        tau = 1.0 / L
        sigma = 1.0 / L
        if verbose:
            print(f"Using step sizes: tau={tau:.4f}, sigma={sigma:.4f}")
    
    losses = []
    
    # Define TV function for HJ-Prox (batch-aware)
    def tv_batch(xb):
        return lambda_1 * compute_tv_pytorch(xb, H, W)
    
    for it in range(iterations):
        x_old = x.clone()
        
        # Dual update: K x_bar
        x_bar_fft = torch.fft.fft2(x_bar)
        Kx = torch.real(torch.fft.ifft2(x_bar_fft * kernel_fft))
        z_update = z + sigma * Kx
        # Closed-form prox for data fidelity (quadratic)
        z = (z_update - sigma * y) / (1.0 + sigma)
        
        # Primal update
        z_fft = torch.fft.fft2(z)
        KTz = torch.real(torch.fft.ifft2(z_fft * kernel_fft_conj))
        x_grad = x - tau * KTz
        x_flat = x_grad.view(-1, 1)
        
        # Compute delta with annealing schedule
        delta_k = 2300000 / (it + 1)**(2 + EPS)
        
        # Proximal step using HJ-Prox for TV
        x_prox, _ = hj_prox(
            x_flat, tau, f=tv_batch, delta=delta_k,
            num_samples=num_samples, alpha=1.0
        )
        
        x = x_prox.view(H, W)
        
        # Over-relaxation
        x_bar = x + theta * (x - x_old)
        
        # Monitor convergence
        if it % 1 == 0:
            x_fft = torch.fft.fft2(x)
            Kx_curr = torch.real(torch.fft.ifft2(x_fft * kernel_fft))
            data_term = 0.5 * torch.sum((Kx_curr - y)**2).item()
            tv_term = compute_tv_pytorch(x.unsqueeze(0).view(1, -1), H, W).item()
            loss = data_term + lambda_1 * tv_term
            losses.append(loss)
            
            if verbose:
                diff = torch.norm(x - x_old).item()
                print(f"Iter {it}: Loss={loss:.6f}, Δx={diff:.6f}")
    
    return x.cpu().numpy(), losses


print("✓ All algorithms and helper functions loaded successfully")

## Problem definition (blurred image)


In [ ]:
# ============================================================================
# CHUNK 2: DATA GENERATION
# ============================================================================

print("\n" + "="*60)
print("Creating synthetic test image and generating data...")
print("="*60)

# Image dimensions
H, W = 64, 64

# Create test image with rectangles
original_img = np.zeros((H, W))
original_img[10:20, 10:30] = 1.0
original_img[30:45, 20:40] = 0.7
original_img[15:35, 35:55] = 0.5

# Parameters
kernel_size = 5
blur_sigma = 0.2
noise_level = 0.15
lambda_1 = 0.125
iterations = 20000

# Create blurred image
print(f"Applying Gaussian blur (kernel size={kernel_size}, sigma={blur_sigma})...")
blurred_img, blur_kernel = blur_image(original_img, kernel_size, blur_sigma)

# Add noise
if noise_level > 0:
    np.random.seed(42)
    blurred_img += noise_level * np.random.randn(*blurred_img.shape)
    blurred_img = np.clip(blurred_img, 0, 1)
    print(f"Added Gaussian noise (σ={noise_level})")

print(f"✓ Data generated: Image size {H}x{W}")
print(f"✓ Regularization parameter λ = {lambda_1}")

## Algorithm 1 — Chambolle–Pock (analytical)


In [ ]:

# ============================================================================
# CHUNK 3: RUN ALGORITHM 1 - Analytical PDHG (Chambolle-Pock)
# ============================================================================

print("\n" + "="*60)
print("Running Algorithm 1: PDHG with Analytical Operators...")
print("="*60)

start_time = time.time()

deblurred_img_Analytical, analytical_losses = chambolle_pock_tv_deblur(
    blurred_img,
    blur_kernel,
    lambda_1=lambda_1,
    iterations=20000,
    tau=0.025*0.0088,
    sigma=0.1,
    theta=1.0,
    verbose=True,
    device='cpu'
)

elapsed_time = time.time() - start_time

# Clip for visualization and metrics
deblurred_clipped = np.clip(deblurred_img_Analytical, 0, 1)

# Compute metrics
mse_blurred = np.mean((original_img - blurred_img)**2)
mse_deblurred = np.mean((original_img - deblurred_clipped)**2)

psnr_blurred = 20 * np.log10(1.0 / np.sqrt(mse_blurred)) if mse_blurred > 0 else float('inf')
psnr_deblurred = 20 * np.log10(1.0 / np.sqrt(mse_deblurred)) if mse_deblurred > 0 else float('inf')

print(f"\n✓ Analytical PDHG completed")
print(f"  - Runtime: {elapsed_time:.2f} seconds")
print(f"  - Final objective: {analytical_losses[-1]:.6f}")
print(f"  - PSNR (blurred): {psnr_blurred:.2f} dB")
print(f"  - PSNR (deblurred): {psnr_deblurred:.2f} dB")
print(f"  - PSNR improvement: {psnr_deblurred - psnr_blurred:.2f} dB")

## Algorithm 2 — PDHG with HJ-Prox


In [ ]:
# ============================================================================
# CHUNK 4: RUN ALGORITHM 2 - PDHG with HJ-Prox
# ============================================================================

print("\n" + "="*60)
print("Running Algorithm 2: PDHG with HJ-Prox...")
print("="*60)

start_time = time.time()

deblurred_img_HJ, hj_losses = pdhg_hjprox_deblur(
    blurred_img,
    blur_kernel,
    lambda_1=lambda_1,
    iterations=20000,
    tau=0.025*0.0088,
    sigma=0.1,
    theta=1.0,
    num_samples=1000,
    delta=0.0001,
    verbose=True,
    device='cpu',
    init_img=blurred_img
)

elapsed_time = time.time() - start_time

print(f"\n✓ PDHG-HJ completed")
print(f"  - Runtime: {elapsed_time:.2f} seconds")
print(f"  - Final objective: {hj_losses[-1]:.6f}")


## Comparison: PDHG vs PDHG-HJ


In [ ]:
import os
os.makedirs('figures', exist_ok=True)
# ============================================================================
# CHUNK 5: GENERATE FIGURES
# ============================================================================

print("\n" + "="*60)
print("Generating figures...")
print("="*60)

# Global font settings
plt.rcParams.update({'font.size': 16})

# --- Figure 1: Original Image ---
plt.figure(figsize=(11, 10))
plt.imshow(original_img, cmap='gray')
plt.title("Original Image", fontsize=40, pad=8)
plt.axis('off')
plt.tight_layout()
plt.savefig('figures/pdhg_original.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 2: PDHG-HJ Deblurred Image ---
plt.figure(figsize=(11, 10))
plt.imshow(deblurred_img_HJ, cmap='gray')
plt.title("PDHG-HJ", fontsize=40, pad=8)
plt.axis('off')
plt.tight_layout()
plt.savefig('figures/pdhg_hj_deblurred.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 3: PDHG Analytical Deblurred Image ---
plt.figure(figsize=(11, 10))
plt.imshow(deblurred_clipped, cmap='gray')
plt.title("PDHG", fontsize=40, pad=8)
plt.axis('off')
plt.tight_layout()
plt.savefig('figures/pdhg_analytical_deblurred.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 4: Objective Function Convergence ---
plt.rcParams.update({'font.size': 20})
plt.figure(figsize=(11, 10))
plt.semilogy(hj_losses, '-', linewidth=3, label=f'PDHG-HJ: {hj_losses[-1]:.3f}')
plt.semilogy(analytical_losses, '--', linewidth=3, label=f'PDHG: {analytical_losses[-1]:.3f}')
plt.title("PDHG Convergence", fontsize=40)
plt.ylabel("Objective (log scale)", fontsize=40)
plt.xlabel("Iteration", fontsize=40)
plt.legend(fontsize=40, loc='upper left')
plt.grid(True, which='both', alpha=0.3)
plt.gca().tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout(pad=2.0)
plt.savefig('figures/pdhg_convergence.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

print("✓ All figures saved: pdhg_original.pdf, pdhg_hj_deblurred.pdf, pdhg_analytical_deblurred.pdf, pdhg_convergence.pdf")
print("\n" + "="*60)
print("✓ ALL EXPERIMENTS COMPLETED SUCCESSFULLY")
print("="*60)